# EDGECAST ? Team 29 A2 Full Reproducible Demo

**Authors:** Suganthini C, Priyam Trivedi  
**Purpose:** Run the actual EDGECAST simulator/controller used by the faculty demo.  

This notebook does **not** use hard-coded result tables. It installs the public submission package, verifies the test suite, loads the real processed FineServe/BurstGPT traces included with the submission, and executes the Nguyen comparator and stabilized cache-aware EDGECAST controller.

In [ ]:
!rm -rf edgecast-submission
!git clone -q https://github.com/priyam1206/edgecast-submission.git
%cd edgecast-submission
!python -m pip install -q -r requirements.txt
print(''EDGECAST environment ready.'')

## 1. Verify the implementation

In [ ]:
!python -m pytest -q

## 2. Run the real FineServe trace
The following cell loads the included processed FineServe Lite trace and runs both controllers on the same selected burst window and the same initial allocation.

In [ ]:
import math
import pandas as pd
from pathlib import Path
from edgecast.datasets import load_preprocessed_trace, select_bursty_window, demand_from_window
from edgecast.policies import CacheAwarePolicy, NguyenPaperPolicy, optimize_initial_allocation
from edgecast.simulation import SimulationConfig, run_simulation

trace = load_preprocessed_trace(Path(''data/preprocessed/FineServe_lite_60s.csv''))
start, end = select_bursty_window(trace, window_slots=40)
demand, models, _ = demand_from_window(trace, start_slot=start, window_slots=40, mode=''requests'')

node_count = 6
total_per_slot = [sum(slot.values()) for slot in demand]
p70 = float(pd.Series(total_per_slot).quantile(.70)) if total_per_slot else 1.0
base_capacity = max(1, int(math.ceil((p70 / max(1, node_count - 1)) * 0.85)))
factors = [.74, .88, 1.0, 1.14, 1.31, .81][:node_count]
node_capacities = tuple(max(1, int(round(base_capacity * factors[i]))) for i in range(node_count))
allocation = optimize_initial_allocation(demand, models, node_capacities)

config = SimulationConfig(
    models=models,
    initial_allocation=tuple(allocation),
    node_capacity=base_capacity,
    node_capacities=node_capacities,
    switch_time=4,
    warm_switch_time=1,
    prefetch_time=4,
    cache_slots=2,
    max_parallel_prefetch=1,
    cooldown=6,
    max_drain_slots=1000,
)

base = run_simulation(demand, NguyenPaperPolicy(max_exact_nodes=8), config)
edge = run_simulation(demand, CacheAwarePolicy(max_exact_nodes=8, min_reallocation_interval=6), config)

pd.DataFrame([base.summary(), edge.summary()])

## 3. Inspect live controller decisions

In [ ]:
rows = []
for i, d in enumerate(edge.decisions):
    if d and (d.get(''action'') != ''hold'' or d.get(''stability_gate'')):
        rows.append({
            ''slot'': i,
            ''action'': d.get(''action''),
            ''Tp'': d.get(''Tp''),
            ''Tc'': d.get(''Tc''),
            ''sigma'': d.get(''sigma''),
            ''net_benefit'': d.get(''net_benefit''),
            ''stability_gate'': d.get(''stability_gate''),
            ''reason'': d.get(''reason''),
        })
pd.DataFrame(rows).head(30)

## 4. Export the current run
The exported files are generated from the simulation run above.

In [ ]:
import json
manifest = {
    ''dataset'': ''FineServe Lite processed trace'',
    ''window'': [int(start), int(end)],
    ''models'': list(models),
    ''base_summary'': base.summary(),
    ''edgecast_summary'': edge.summary(),
}
with open(''edgecast_colab_run.json'', ''w'') as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))